# Task 1: Preprocessing

**Input:** `data/raw/reviews_raw.csv` — raw scraped reviews  
**Output:** `data/clean/reviews_clean.csv` — translated, deduplicated, normalized dataset

**Steps:**
1. Load raw data and inspect
2. Translate Amharic reviews to English (emojis preserved)
3. Remove duplicate reviews
4. Handle missing values
5. Normalize dates to `YYYY-MM-DD`
6. Final quality check and export

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from scripts.preprocess import translate_amharic, preprocess_reviews

## 1. Load Raw Data

In [2]:
raw_df = pd.read_csv("../data/raw/reviews_raw.csv")

print(f"Shape: {raw_df.shape}")
print(f"\nColumns: {list(raw_df.columns)}")
print(f"\nDtypes:\n{raw_df.dtypes}")
raw_df.head()

Shape: (1800, 6)

Columns: ['id', 'review', 'rating', 'date', 'bank', 'source']

Dtypes:
id        object
review    object
rating     int64
date      object
bank      object
source    object
dtype: object


,id,review,rating,date,bank,source
0,06f6640c-b65c-43e4-88ef-0a79be8b9534,it's a good application,5,2026-05-13 20:28:58,Commercial Bank of Ethiopia,Google Play
1,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,Commercial Bank of Ethiopia,Google Play
2,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,Commercial Bank of Ethiopia,Google Play
3,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13 12:19:17,Commercial Bank of Ethiopia,Google Play
4,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13 11:27:52,Commercial Bank of Ethiopia,Google Play


## 2. Pre-cleaning Snapshot

In [3]:
print("=== Reviews per bank (raw) ===")
print(raw_df["bank"].value_counts())

print("\n=== Null values per column ===")
print(raw_df.isnull().sum())

print("\n=== Duplicate review IDs ===")
print(f"Total duplicates: {raw_df.duplicated(subset='id').sum()}")

=== Reviews per bank (raw) ===
bank
Commercial Bank of Ethiopia    600
Bank of Abyssinia              600
Dashen Bank                    600
Name: count, dtype: int64

=== Null values per column ===
id        0
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

=== Duplicate review IDs ===
Total duplicates: 0


## 3. Translate Amharic Reviews

Many users write reviews in Amharic. We detect these using `langdetect` and translate them to English using `deep-translator`'s Google Translate backend. Emojis are preserved since they carry sentiment signal.

Only reviews detected as Amharic (`lang == 'am'`) are sent to the translation API — English reviews are left untouched.

In [4]:
translated_df = translate_amharic(raw_df.copy())
translated_df.head()

Translating Amharic reviews: 100%|██████████| 1800/1800 [00:56<00:00, 31.64it/s]


[translate] Amharic reviews translated: 78


,id,review,rating,date,bank,source
0,06f6640c-b65c-43e4-88ef-0a79be8b9534,it's a good application,5,2026-05-13 20:28:58,Commercial Bank of Ethiopia,Google Play
1,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,Commercial Bank of Ethiopia,Google Play
2,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,Commercial Bank of Ethiopia,Google Play
3,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13 12:19:17,Commercial Bank of Ethiopia,Google Play
4,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13 11:27:52,Commercial Bank of Ethiopia,Google Play


## 4. Clean & Normalize

Calling `preprocess_reviews()` from `scripts/preprocess.py` which:
- Drops duplicate `id` entries
- Drops rows where `review` or `rating` is null
- Normalizes `date` to `YYYY-MM-DD`
- Returns only the five required columns: `review`, `rating`, `date`, `bank`, `source`

In [5]:
clean_df = preprocess_reviews(translated_df)
clean_df.head()

[preprocess] Original rows     : 1800
[preprocess] Duplicates removed: 0
[preprocess] Nulls removed     : 0
[preprocess] Final row count   : 1800


,review,rating,date,bank,source
0,it's a good application,5,2026-05-13,Commercial Bank of Ethiopia,Google Play
1,thank you cbe,5,2026-05-13,Commercial Bank of Ethiopia,Google Play
2,is good,5,2026-05-13,Commercial Bank of Ethiopia,Google Play
3,wow,5,2026-05-13,Commercial Bank of Ethiopia,Google Play
4,Good application,2,2026-05-13,Commercial Bank of Ethiopia,Google Play


## 5. Post-cleaning Quality Check

In [6]:
print("=== Reviews per bank (clean) ===")
print(clean_df["bank"].value_counts())

print("\n=== Missing values ===")
print(clean_df.isnull().sum())

print("\n=== Date range ===")
print(f"Earliest: {clean_df['date'].min()}")
print(f"Latest:   {clean_df['date'].max()}")

print("\n=== Rating distribution ===")
print(clean_df["rating"].value_counts().sort_index())

print(f"\n=== Total reviews: {len(clean_df)} ===")

=== Reviews per bank (clean) ===
bank
Commercial Bank of Ethiopia    600
Bank of Abyssinia              600
Dashen Bank                    600
Name: count, dtype: int64

=== Missing values ===
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

=== Date range ===
Earliest: 2024-11-12
Latest:   2026-05-13

=== Rating distribution ===
rating
1     387
2      61
3      98
4     134
5    1120
Name: count, dtype: int64

=== Total reviews: 1800 ===


## 6. Export Clean Dataset

Saved to `data/clean/reviews_clean.csv`. Listed in `.gitignore` — will not be committed to GitHub.

In [7]:
os.makedirs("../data/clean", exist_ok=True)

# Save individual bank files
for bank_name, group in clean_df.groupby("bank"):
    filename = bank_name.lower().replace(" ", "_") + "_clean.csv"
    group.to_csv(f"../data/clean/{filename}", index=False)
    print(f"Saved {len(group)} reviews → data/clean/{filename}")

# Save combined file for use in tasks 2, 3, and 4
clean_df.to_csv("../data/clean/reviews_clean.csv", index=False)
print(f"\nSaved combined {len(clean_df)} reviews → data/clean/reviews_clean.csv")

Saved 600 reviews → data/clean/bank_of_abyssinia_clean.csv
Saved 600 reviews → data/clean/commercial_bank_of_ethiopia_clean.csv
Saved 600 reviews → data/clean/dashen_bank_clean.csv

Saved combined 1800 reviews → data/clean/reviews_clean.csv
